# 2.4. Perform nested regression with bootstrapping of metric value against magnitude of ablation and biological covariate confluence

Assesses how each metrics is able to capture the variation in ablation magnitude while
being unbiased across biological covariates.

In [1]:
import pathlib
from typing import Optional

import pandas as pd
import polars as pl

from image_ablation_analysis.regression.nested_regression import (
    bootstrap_nested_regression,
    BootstrapConfig,
    ColumnSpec,
)

## Pathing

In [2]:
results_dir = pathlib.Path(".") / "results"
if not results_dir.exists():
    raise FileNotFoundError(f"Results directory not found at {results_dir.resolve()}")

regression_input_data_file = results_dir / "for_analysis_subsampled.parquet"
if not regression_input_data_file.exists():
    raise FileNotFoundError(f"Regression input data not found at {regression_input_data_file.resolve()}")

## Regression helper

In [3]:
def summarize_r2_scatter_bootstrap(
    boot_df: pd.DataFrame,
    output_csv: Optional[str | pathlib.Path] = None,
    group_cols: tuple[str, ...] = ("metric_name", "ablation_type"),
    restricted_col: str = "r2_restricted",
    partial_col: str = "partial_r2_x2",
    ci: float = 0.95,
) -> pd.DataFrame:
    
    required = set(group_cols) | {"boot_idx", restricted_col, partial_col}
    missing = sorted(required - set(boot_df.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    lower_q = (1 - ci) / 2
    upper_q = 1 - lower_q

    df = boot_df.copy()
    df[restricted_col] = pd.to_numeric(df[restricted_col], errors="coerce")
    df[partial_col] = pd.to_numeric(df[partial_col], errors="coerce")

    summary = (
        df.groupby(list(group_cols), dropna=False)
        .agg(
            n_boot=("boot_idx", "nunique"),

            restricted_r2_mean=(restricted_col, "mean"),
            restricted_r2_lower=(restricted_col, lambda x: x.quantile(lower_q)),
            restricted_r2_upper=(restricted_col, lambda x: x.quantile(upper_q)),

            partial_r2_mean=(partial_col, "mean"),
            partial_r2_lower=(partial_col, lambda x: x.quantile(lower_q)),
            partial_r2_upper=(partial_col, lambda x: x.quantile(upper_q)),
        )
        .reset_index()
        .sort_values(list(group_cols))
        .reset_index(drop=True)
    )

    if output_csv is not None:
        output_csv = pathlib.Path(output_csv)
        output_csv.parent.mkdir(parents=True, exist_ok=True)
        summary.to_csv(output_csv, index=False)

    return summary

In [4]:
regression_input = pl.read_parquet(regression_input_data_file).to_pandas()
print(len(regression_input))
regression_input.head()

1844400


,created_at,run_id,original_abs_path,original_rel_path,aug_abs_path,aug_rel_path,variant,config_id,params_json,param_fixed,...,Metadata_PositionX,Metadata_PositionY,Metadata_PositionZ,Metadata_Row,Metadata_Reimaged,ablation_package,ablation_type,hash,metric_name,metric_value
0,20260218T212130127579Z,0f57f65f-729e-4cc4-a651-121c34b36181,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gaussnoise=0.2_6339a071e1e884db,albumentations:GaussNoise:6339a071e1e884db,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,GaussNoise,6339a071e1e884db,lpips,0.712087
1,20260219T025541654122Z,41811ddb-a68d-4909-830b-cee3d15b03ee,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gamma=155.18_699b97b2f9705fc2,albumentations:RandomGamma:699b97b2f9705fc2,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,RandomGamma,699b97b2f9705fc2,foreground_ssim,0.457415
2,20260219T025541654122Z,41811ddb-a68d-4909-830b-cee3d15b03ee,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gamma=193.32_0c77f398c28c2341,albumentations:RandomGamma:0c77f398c28c2341,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,RandomGamma,0c77f398c28c2341,ssim,0.714855
3,20260219T025541654122Z,41811ddb-a68d-4909-830b-cee3d15b03ee,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gamma=155.18_699b97b2f9705fc2,albumentations:RandomGamma:699b97b2f9705fc2,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,RandomGamma,699b97b2f9705fc2,mae,0.006020
4,20260218T212130127579Z,0f57f65f-729e-4cc4-a651-121c34b36181,/mnt/data_nvme1/data/ALSF_pilot_data/SN0313537...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,/mnt/hdd20tb/alsf_ablation/SN0313537/BR0014397...,SN0313537/BR00143976__2024-07-04T16_04_45-Meas...,xform_abl_gaussnoise=0.1_e473bcd5b5024cf6,albumentations:GaussNoise:e473bcd5b5024cf6,"{""backend"":""albumentations"",""transform_name"":""...",[],...,-0.000646,0.000646,-0.000006,3.0,False,albumentations,GaussNoise,e473bcd5b5024cf6,foreground_ssim,0.143215


## Shared boostrap/regression parameters
All regression analysis will share the same dependent variable, whichare the metric values as well as the first (restricted) independent variable which will be the parameter value. The full independent variable and the groupings of regression analysis will change based on the confounding variable being tested for.

In [5]:
regression_config = {
    "y": "metric_value",    # dependent variable, always metric value for this analysis
    "x1": "param_values",   # independent variable 1, always the ablation parameter values for this analysis
}

bootstrap_config = {
    "n_boot": 300,
    "sample_frac": 0.5,
    "replace": True,
    "standardize": False,
    "robust_cov": None,     # or "HC3"
    "min_group_size": 25,   # prevent regression on tiny groups
}

## Regression Analysis 1: Assessing confounding by seeding density

In [6]:
colspec = ColumnSpec(
    group_cols=("metric_name", "ablation_type"),
    x2="seeding_density", # full regression parameters
    x2_categorical=False,
    standardize_cols=("param_values", "seeding_density"),
    **regression_config
)

cfg = BootstrapConfig(
    **bootstrap_config
)

boot_res = bootstrap_nested_regression(regression_input, colspec, cfg)
boot_res.to_parquet(results_dir / "boot_nest_confluence.parquet", index=False)

Bootstrap groups:   0%|          | 0/42 [00:00<?, ?it/s]

In [7]:
summarize_r2_scatter_bootstrap(
    boot_res,
    output_csv=results_dir / "boot_nest_confluence_summary.csv",
)

,metric_name,ablation_type,n_boot,restricted_r2_mean,restricted_r2_lower,restricted_r2_upper,partial_r2_mean,partial_r2_lower,partial_r2_upper
0,dists,Dilate,300,0.333146,0.322511,0.344332,0.077436,6.993122e-02,0.084361
1,dists,Erode,300,0.448721,0.438195,0.459475,0.092615,8.521768e-02,0.100562
2,dists,GaussNoise,300,0.112250,0.104277,0.121178,0.059226,5.244090e-02,0.066842
3,dists,GaussianBlur,300,0.170199,0.161275,0.179939,0.076169,6.861769e-02,0.082725
4,dists,GridDistortion,300,0.672356,0.664987,0.679806,0.029703,2.538851e-02,0.034636
5,dists,RandomGamma,300,0.202411,0.194180,0.210199,0.000046,5.179250e-08,0.000210
6,foreground_psnr,Dilate,300,0.206450,0.197217,0.218368,0.000571,9.137378e-05,0.001383
7,foreground_psnr,Erode,300,0.107291,0.099907,0.116198,0.000137,7.851101e-07,0.000571
8,foreground_psnr,GaussNoise,300,0.889034,0.886595,0.891238,0.052885,4.649919e-02,0.059514
9,foreground_psnr,GaussianBlur,300,0.043879,0.038544,0.049774,0.000061,1.315380e-07,0.000323


## Regression Analysis 2: Assessing confounding by cell lines

In [8]:
colspec = ColumnSpec(
    group_cols=("metric_name", "ablation_type"),
    x2="cell_line", # categorical var
    x2_categorical=True,
    standardize_cols=("param_values",),
    **regression_config
)

cfg = BootstrapConfig(
    **bootstrap_config
)

boot_res = bootstrap_nested_regression(regression_input, colspec, cfg)
boot_res.to_parquet(results_dir / "boot_nest_cell_line.parquet", index=False)

Bootstrap groups:   0%|          | 0/42 [00:00<?, ?it/s]

In [9]:
summarize_r2_scatter_bootstrap(
    boot_res,
    output_csv=results_dir / "boot_nest_cell_line_summary.csv",
)

,metric_name,ablation_type,n_boot,restricted_r2_mean,restricted_r2_lower,restricted_r2_upper,partial_r2_mean,partial_r2_lower,partial_r2_upper
0,dists,Dilate,300,0.333146,0.322511,0.344332,0.056038,0.049346,0.062477
1,dists,Erode,300,0.448721,0.438195,0.459475,0.068488,0.062534,0.074851
2,dists,GaussNoise,300,0.112250,0.104277,0.121178,0.034945,0.030441,0.041503
3,dists,GaussianBlur,300,0.170199,0.161275,0.179939,0.033587,0.028624,0.039012
4,dists,GridDistortion,300,0.672356,0.664987,0.679806,0.030518,0.025575,0.035574
5,dists,RandomGamma,300,0.202411,0.194180,0.210199,0.018191,0.015789,0.020638
6,foreground_psnr,Dilate,300,0.206450,0.197217,0.218368,0.265293,0.254779,0.274316
7,foreground_psnr,Erode,300,0.107291,0.099907,0.116198,0.171184,0.160786,0.180781
8,foreground_psnr,GaussNoise,300,0.889034,0.886595,0.891238,0.048141,0.042142,0.054292
9,foreground_psnr,GaussianBlur,300,0.043879,0.038544,0.049774,0.210995,0.198517,0.222304
